In [1]:
# ===========================================
# Libraries
# ===========================================

import numpy as np
from scipy.stats import gaussian_kde

In [2]:
# ===========================================
# Load chains
# ===========================================

path_base = "/home/brunowesley/projetos/MCMC-cosmo/Codes/GR-based/CPL/CC_SNe_BAO/flat_samples_CPL_cc_sne_bao.npy"
path_frb  = "/home/brunowesley/projetos/MCMC-cosmo/Codes/GR-based/CPL/CC_SNe_BAO_FRB/flat_samples_CPL_cc_sne_bao_frb.npy"

samples_base = np.load(path_base)
samples_frb  = np.load(path_frb)

# ===========================================
# Extract parameters (H0: index 0, Ob: index 2)
# ===========================================

H0_base, Ob_base = samples_base[:, 0], samples_base[:, 2]
H0_frb,  Ob_frb  = samples_frb[:, 0],  samples_frb[:, 2]

In [3]:
# ===========================================
# Figure of Merit via 2D KDE (HPD-based)
# ===========================================

def get_fom_kde(x, y, credible_level=0.683, grid_pts=300, max_samples=5000, padding=0.20, random_state=42):
    
    # Reproducible subsampling
    rng = np.random.default_rng(random_state)

    n = len(x)

    if n > max_samples:
        idx = rng.choice(n, size=max_samples, replace=False)
        x_kde = x[idx]
        y_kde = y[idx]
    else:
        x_kde = x
        y_kde = y

    # KDE fit
    data = np.vstack([x_kde, y_kde])
    kde = gaussian_kde(data)

    # Enlarged grid to include KDE tails
    dx = padding * (x.max() - x.min())
    dy = padding * (y.max() - y.min())

    x_grid = np.linspace(
        x.min() - dx,
        x.max() + dx,
        grid_pts
    )

    y_grid = np.linspace(
        y.min() - dy,
        y.max() + dy,
        grid_pts
    )

    X, Y = np.meshgrid(x_grid, y_grid)

    Z = kde(
        np.vstack([X.ravel(), Y.ravel()])
    ).reshape(grid_pts, grid_pts)

    # Cell area
    dA = (x_grid[1] - x_grid[0]) * (y_grid[1] - y_grid[0])

    # Explicit normalization
    Z /= np.sum(Z) * dA

    # HPD threshold
    Z_sorted = np.sort(Z.ravel())[::-1]
    cumulative = np.cumsum(Z_sorted) * dA
    idx_thr = np.searchsorted(cumulative, credible_level)
    idx_thr = min(idx_thr, len(Z_sorted) - 1)
    threshold = Z_sorted[idx_thr]

    # HPD area
    area = np.sum(Z >= threshold) * dA
    area = max(area, np.finfo(float).eps)

    # Figure of Merit
    fom = 1.0 / area

    return fom, area


# ===========================================
# Summary statistics (non-Gaussian safe)
# ===========================================

def get_stats(x, y):
    
    # FoM via KDE
    fom, area = get_fom_kde(x, y)

    # Effective 1σ uncertainty from percentiles
    def sigma_eff(s):
        low, _, high = np.percentile(s, [16, 50, 84])
        return 0.5 * (high - low)

    # Third standardized moment
    def skewness(s):
        sigma = np.std(s)
        if sigma == 0:
            return 0.0
        return np.mean((s - np.mean(s))**3) / sigma**3

    return (
        fom,
        area,
        sigma_eff(x),
        sigma_eff(y),
        skewness(x),
        skewness(y),
    )


# ===========================================
# Compute results
# ===========================================

fom_b, area_b, sH0_b, sOb_b, skH0_b, skOb_b = get_stats(H0_base, Ob_base)
fom_f, area_f, sH0_f, sOb_f, skH0_f, skOb_f = get_stats(H0_frb,  Ob_frb)

# Relative FoM improvement from adding FRBs
improvement_fom = (fom_f / fom_b - 1.0) * 100.0

# Precision gain on each parameter
gain_H0 = (1.0 - sH0_f / sH0_b) * 100.0
gain_Ob  = (1.0 - sOb_f / sOb_b) * 100.0


# ===========================================
# Print results
# ===========================================

print("\n" + "="*65)
print(f"{'CPL HPD-KDE Figure of Merit Analysis':^65}")
print("="*65)

print("\n--- CC + SNe + BAO (Baseline) ---")
print(f"FoM_HPD              = {fom_b:.4f}")
print(f"HPD area (68.3%)     = {area_b:.6f}")
print(f"Eff. σ(H0) [68%]     = {sH0_b:.3f} km/s/Mpc")
print(f"Eff. σ(Ωb) [68%]     = {sOb_b:.5f}")
print(f"Skew(H0)             = {skH0_b:.3f}")
print(f"Skew(Ωb)             = {skOb_b:.3f}")

print("\n--- CC + SNe + BAO + FRB ---")
print(f"FoM_HPD              = {fom_f:.4f}")
print(f"HPD area (68.3%)     = {area_f:.6f}")
print(f"Eff. σ(H0) [68%]     = {sH0_f:.3f} km/s/Mpc")
print(f"Eff. σ(Ωb) [68%]     = {sOb_f:.5f}")
print(f"Skew(H0)             = {skH0_f:.3f}")
print(f"Skew(Ωb)             = {skOb_f:.3f}")

print("\n--- Comparison ---")
print(f"FoM_HPD improvement  = {improvement_fom:.2f}%")
print(f"H0 precision gain    = {gain_H0:.2f}%")
print(f"Ωb precision gain    = {gain_Ob:.2f}%")
print("="*65)


              CPL HPD-KDE Figure of Merit Analysis               

--- CC + SNe + BAO (Baseline) ---
FoM_HPD              = 17.7194
HPD area (68.3%)     = 0.056435
Eff. σ(H0) [68%]     = 1.563 km/s/Mpc
Eff. σ(Ωb) [68%]     = 0.00504
Skew(H0)             = 0.019
Skew(Ωb)             = 2.391

--- CC + SNe + BAO + FRB ---
FoM_HPD              = 33.7989
HPD area (68.3%)     = 0.029587
Eff. σ(H0) [68%]     = 1.384 km/s/Mpc
Eff. σ(Ωb) [68%]     = 0.00288
Skew(H0)             = -0.077
Skew(Ωb)             = 0.197

--- Comparison ---
FoM_HPD improvement  = 90.75%
H0 precision gain    = 11.47%
Ωb precision gain    = 42.89%
